# qassette.memory calibration

Reproduces the single-qubit coherence measurements of **Bozkurt et al., Nature Physics 21, 1469 (2025)** against `qassette.memory`:

| sequence | protocol | paper value |
|---|---|---|
| Ramsey | prepare $|+\rangle$, wait $t$, measure $\langle X\rangle$, $t \in [0, 300\,\mu s]$ | $T_2^* = 64\,\mu s$ |
| Hahn echo | X pulse at $t/2$, $t \in [0, 3\,ms]$ | $T_{2E} = 562\,\mu s$ |
| CPMG-2 | X pulses at $t/4, 3t/4$ | $T_{2,CP2} = 1.02\,ms$ |
| $T_1$ | prepare $|1\rangle$, wait, measure $P(|1\rangle)$, $t \in [0, 100\,ms]$ | $T_1 = 25\,ms$ |

All curves are measured through `qassette.memory.noisy_vectors` on one qubit. The tuned noise parameters are the package defaults in `qassette.memory.MemoryParams` (written by `scripts/calibrate.py`).

In [ ]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

from calibrate import TARGETS, make_figure, tune, verify
from qassette.memory import MemoryParams

## 1. (Optional) re-run the tuner

Nelder-Mead in log space over $(\sigma_{qs}, a_{tls}, \gamma_{tls})$ on a fast vectorized surrogate of the same noise process (multi-start, ~15 min). Skip this cell to audit the shipped defaults.

In [ ]:
RETUNE = False
if RETUNE:
    sigma, a, gamma = tune()
else:
    d = MemoryParams(t=0.0)  # shipped defaults
    sigma, a, gamma = d.sigma_qs, d.a_tls, d.gamma_tls
print(f"sigma_qs = {sigma:.4g} rad/s | a_tls = {a:.4g} rad/s | gamma_tls = {gamma:.4g} /s")

## 2. Measure all four decays through qassette.memory and check the fits

In [ ]:
times = verify(sigma, a, gamma)
print(f"{'sequence':<10} {'simulated':>12} {'paper':>10} {'deviation':>10}")
for seq, tgt in TARGETS.items():
    dev = times[seq] / tgt - 1
    ok = "within 15%" if abs(dev) <= 0.15 else "OUT OF SPEC"
    print(f"{seq:<10} {times[seq] * 1e6:>10.1f}us {tgt * 1e6:>8.0f}us {dev:>+9.1%}  {ok}")

## 3. Publication figure → docs/calibration.png

In [ ]:
fig = make_figure(sigma, a, gamma, ROOT / "docs" / "calibration.png")
fig

### Physics note

A single telegraph fluctuator plus a quasi-static Gaussian cannot match all three dephasing times *exactly*: in the slow-fluctuator limit the best achievable CPMG-2/Hahn ratio is $4^{1/3} \approx 1.59$, while the paper reports $1.02\,ms / 562\,\mu s = 1.82$ — real devices see an ensemble of TLS with a spread of switching rates. Within the ±15% calibration tolerance, a slow fluctuator ($\gamma_{tls} \cdot T_{2E} \ll 1$) satisfies all four targets simultaneously, which is what the tuner converges to.